In [ ]:
#this explores the output from the smart power 3, left it in so others could explore the dataoutput

voltage = []
current = []
power = []

with open('txt/putty.log', 'r') as log_file:
    with open('output_file.txt', 'w') as output_file:
        # Skip the first line if necessary
        first_line = log_file.readline().strip()
        print(first_line)
        if not first_line.startswith('0000'):
            # Not a data line, skip it
            first_line = log_file.readline().strip()
        prev_line = ''
        for line in log_file:
            print(line)
            # Skip empty lines
            if line.strip() == '':
                continue
            values = line.strip().split(',')
            if len(values) == 15: #sometime the records can glitch with recordings this stops errors from this
                voltage.append(int(values[5]))
                current.append(int(values[6]))
                power.append(int(values[7]))
                output_file.write(f'{values[5]},{values[6]},{values[7]}\n')

print(max(voltage))
print("mV")
print(max(current))
print("mA")
print(max(power))
print("mW")

=~=~=~=~=~=~=~=~=~=~=~= PuTTY log 2023.03.28 15:13:56 =~=~=~=~=~=~=~=~=~=~=~=


0000068624,15408,0045,00000,0,00003,0000,00000,0,0,00003,0000,00000,0,0



0000068724,15391,0055,00000,0,00002,0000,00000,0,0,00001,0000,00000,0,0



0000068824,15391,0055,00000,0,00004,0000,00000,0,0,00004,0000,00000,0,0



0000068924,15391,0055,00000,0,00005,0000,00000,0,0,00001,0000,00000,0,0



0000069024,15401,0047,00000,0,00000,0000,00000,0,0,00000,0000,00000,0,0



0000069124,15401,0047,00000,0,00000,0000,00000,0,0,00000,0000,00000,0,0



0000069224,15401,0047,00000,0,00000,0000,00000,0,0,00001,0000,00000,0,0



0000069324,15387,0046,00000,0,00004,0000,00000,0,0,00000,0000,00000,0,0



0000069424,15387,0046,00000,0,00000,0000,00000,0,0,00000,0000,00000,0,0



0000069524,15387,0046,00000,0,00000,0000,00000,0,0,00000,0000,00000,0,0



0000069624,15414,0047,00000,0,00000,0000,00000,0,0,00000,0000,00000,0,0



0000069724,15414,0047,00000,0,00000,0000,00000,0,0,00000,0000,00000,0,0



0000069824,15414,004

In [ ]:
#Records power from the smart power on command(Due to multiple tests an the interative nature i left in a section for diffrent boards to stop re-writing)
import serial
import time

# Open the serial port
ser = serial.Serial('COM1', 9600, timeout=1)

# Start time
start_time = time.time()

# Read data for 100 seconds
data = b''
while (time.time() - start_time) < 100:
    data += ser.read(ser.in_waiting or 1)

# Save data to file
with open('serial_data.txt', 'wb') as f:
    f.write(data)

# Close the serial port
ser.close()

In [ ]:
#basic connection 
import socket

HOST = 'localhost' # use localhost
PORT = 5000 # choose a port number

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind((HOST, PORT))
    s.listen()
    print(f"Listening on {HOST}:{PORT}")
    conn, addr = s.accept()
    with conn:
        print(f"Connected by {addr}")
        while True:
            data = conn.recv(1024)
            if not data:
                break
            print(f"Received: {data.decode('utf-8')}")

Listening on localhost:5000


In [ ]:
#Records power from the smart power on command(Due to multiple tests an the interative nature i left in a section for diffrent boards to stop re-writing)

import serial
import time

# Open the serial port
ser = serial.Serial('COM5', 921600, timeout=1)

# Start time
start_time = time.time()

# Read data for 100 seconds
data = b''
while (time.time() - start_time) < 100:
    data += ser.read(ser.in_waiting or 1)

# Save data to file
with open('standby-pi3b+.txt', 'wb') as f:
    f.write(data)

# Close the serial port
ser.close()

In [ ]:
#Records power from the smart power on command

import serial
import time

# Open the serial port
ser = serial.Serial('COM5', 921600, timeout=1)

# Start time
start_time = time.time()

# Read data for 100 seconds
data = b''
while (time.time() - start_time) < 100:
    data += ser.read(ser.in_waiting or 1)

# Save data to file
with open('odroidxu4.txt', 'wb') as f:
    f.write(data)

# Close the serial port
ser.close()

In [ ]:
#Records power when signal sent from edge board, when looking at power,current and voltage over the whole process 

import serial
import time
import threading
import socket
global smartpowerdata
smartpowerdata = b''
# configure serial port
global ser 
ser = serial.Serial('COM5', 921600)

# flag to indicate whether to record data
record_data = False
write_data = False
# function to record data from serial port
def record_serial():
    global f
    global datapower
    datapower = b''
    while True:
        if record_data:
            f = open(filename+"serial_data.txt", "a")
            line = ser.readline().decode()
            f.write(line+ "\n")
            f.flush()
            f.close()

# Save data to file
# create and configure socket connection
HOST = '0.0.0.0' 
PORT = 5000  # can be any port number

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.bind((HOST, PORT))
s.listen(1)

# function to handle incoming connections
def handle_connection(conn):
    global record_data, filename
    global write_data
    while True:
        data = conn.recv(1024).decode()
        if not data:
            break
        if data.startswith("start"):
            #f = open("serial_data.txt", "w")
            #filename = data.split('/')[1]
            filename= data
            ser.reset_input_buffer()
            record_data = True
            print("Recording started")
        elif data == "stop":
            write_data = True
            record_data = False
            print("Recording stopped")
    conn.close()

# function to start socket server and record data
def start_server():
    # start recording thread
    t = threading.Thread(target=record_serial)
    t.start()

    # start socket server
    while True:
        conn, addr = s.accept()
        print('Connected by', addr)
        threading.Thread(target=handle_connection, args=(conn,)).start()

# start socket server and record data
start_server()

Connected by ('192.168.0.107', 55398)
Recording started
Connected by ('192.168.0.107', 55406)
Recording stopped
Connected by ('192.168.0.107', 55422)
Recording started
Connected by ('192.168.0.107', 55432)
Recording stopped
Connected by ('192.168.0.107', 55442)
Recording started
Connected by ('192.168.0.107', 55444)
Recording stopped
Connected by ('192.168.0.107', 55452)
Recording started
Connected by ('192.168.0.107', 55462)
Recording stopped
Connected by ('192.168.0.107', 55476)
Recording started
Connected by ('192.168.0.107', 55482)
Recording stopped
Connected by ('192.168.0.107', 55496)
Recording started
Connected by ('192.168.0.107', 55502)
Recording stopped
Connected by ('192.168.0.107', 55518)
Recording started
Connected by ('192.168.0.107', 55522)
Recording stopped
Connected by ('192.168.0.107', 55526)
Recording started
Connected by ('192.168.0.107', 55528)
Recording stopped
Connected by ('192.168.0.107', 55536)
Recording started
Connected by ('192.168.0.107', 55548)
Recording 

Exception in thread Thread-3 (record_serial):
Traceback (most recent call last):
  File "c:\Users\jspre\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\jspre\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\jspre\AppData\Local\Temp\ipykernel_12276\233406444.py", line 21, in record_serial
  File "c:\Users\jspre\AppData\Local\Programs\Python\Python310\lib\site-packages\IPython\core\interactiveshell.py", line 282, in _modified_open
    return io_open(file, *args, **kwargs)
FileNotFoundError: [Errno 2] No such file or directory: 'start yolov5models/yolov5s-diorship-int8inferserial_data.txt'


Connected by ('192.168.0.107', 56502)
Recording started
Connected by ('192.168.0.107', 56510)
Recording stopped
Connected by ('192.168.0.107', 56516)
Recording started
Connected by ('192.168.0.107', 56526)
Recording stopped
Connected by ('192.168.0.107', 56540)
Recording started
Connected by ('192.168.0.107', 56548)
Recording stopped
Connected by ('192.168.0.107', 56550)
Recording started
Connected by ('192.168.0.107', 56566)
Recording stopped
Connected by ('192.168.0.107', 56570)
Recording started
Connected by ('192.168.0.107', 56572)
Recording stopped
Connected by ('192.168.0.107', 56582)
Recording started
Connected by ('192.168.0.107', 56584)
Recording stopped
Connected by ('192.168.0.107', 56588)
Recording started
Connected by ('192.168.0.107', 56592)
Recording stopped
Connected by ('192.168.0.107', 56596)
Recording started
Connected by ('192.168.0.107', 56612)
Recording stopped
Connected by ('192.168.0.107', 56614)
Recording started
Connected by ('192.168.0.107', 56616)
Recording 

In [ ]:
#Records power when signal sent from edge board, when only looking at inference power,current and voltage use this signal records
import serial
import time
import threading
import socket
global smartpowerdata
smartpowerdata = b''
# configure serial port
global ser 
ser = serial.Serial('COM5', 921600)

# flag to indicate whether to record data
record_data = False
write_data = False
# function to record data from serial port
def record_serial():
    global f
    global datapower
    datapower = b''
    while True:
        if record_data:
            f = open(filename+"serial_data.txt", "a")
            line = ser.readline().decode()
            f.write(line+ "\n")
            f.flush()
            f.close()

# Save data to file
# create and configure socket connection
HOST = '0.0.0.0' 
PORT = 5000  # can be any port number

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.bind((HOST, PORT))
s.listen(1)

# function to handle incoming connections
def handle_connection(conn):
    global record_data, filename
    global write_data
    while True:
        data = conn.recv(1024).decode()
        if not data:
            break
        if data.startswith("start"):
            #f = open("serial_data.txt", "w")
            if '/' in data:
                filename = data.split('/')[1]
            else:
                filename = data
            #filename= data
            ser.reset_input_buffer()
            record_data = True
            print("Recording started")
        elif data == "stop":
            write_data = True
            record_data = False
            print("Recording stopped")
    conn.close()

# function to start socket server and record data
def start_server():
    # start recording thread
    t = threading.Thread(target=record_serial)
    t.start()

    # start socket server
    while True:
        conn, addr = s.accept()
        print('Connected by', addr)
        threading.Thread(target=handle_connection, args=(conn,)).start()

# start socket server and record data
start_server()

Connected by ('192.168.0.107', 58640)
Recording started
Connected by ('192.168.0.107', 58648)
Recording stopped
Connected by ('192.168.0.107', 58656)
Recording started
Connected by ('192.168.0.107', 58664)
Recording stopped
Connected by ('192.168.0.107', 58672)
Recording started
Connected by ('192.168.0.107', 58678)
Recording stopped
Connected by ('192.168.0.107', 58680)
Recording started
Connected by ('192.168.0.107', 58684)
Recording stopped
Connected by ('192.168.0.107', 58694)
Recording started
Connected by ('192.168.0.107', 58696)
Recording stopped
Connected by ('192.168.0.107', 58712)
Recording started
Connected by ('192.168.0.107', 58718)
Recording stopped
Connected by ('192.168.0.107', 58722)
Recording started
Connected by ('192.168.0.107', 58728)
Recording stopped
Connected by ('192.168.0.107', 58738)
Recording started
Connected by ('192.168.0.107', 58750)
Recording stopped
Connected by ('192.168.0.107', 58752)
Recording started
Connected by ('192.168.0.107', 58766)
Recording 